### CEO Knowledge Agents

Creates Agent Bricks Knowledge Assistants for the 4 CEO-specific document corpora:
- **Legal Complaints KA** — employment, liability, vendor disputes
- **Regulatory Documents KA** — permits, fire safety, zoning, FDA
- **Audit Reports KA** — financial, operational, food safety, supply chain audits
- **Consultancy Reports KA** — strategy, ops efficiency, AI transformation, workforce

In [ ]:
%pip install --upgrade databricks-sdk
dbutils.library.restartPython()

In [ ]:
CATALOG = dbutils.widgets.get("CATALOG")
LEGAL_ENDPOINT   = dbutils.widgets.get("CEO_LEGAL_KA_ENDPOINT")
REG_ENDPOINT     = dbutils.widgets.get("CEO_REGULATORY_KA_ENDPOINT")
AUDIT_ENDPOINT   = dbutils.widgets.get("CEO_AUDIT_KA_ENDPOINT")
CONSULT_ENDPOINT = dbutils.widgets.get("CEO_CONSULTANCY_KA_ENDPOINT")

In [ ]:
from databricks.sdk import WorkspaceClient
import json, time, sys

sys.path.append('../utils')
from uc_state import add

w = WorkspaceClient()
API_BASE = "/api/2.0/knowledge-assistants"

KA_CONFIGS = [
    {
        "name": f"{CATALOG}-ceo-legal",
        "endpoint": LEGAL_ENDPOINT,
        "volume_path": f"/Volumes/{CATALOG}/legal_complaints/documents",
        "source_name": "legal_complaint_documents",
        "description": (
            "Answers CEO-level questions about Casper's Kitchens legal exposure. "
            "Covers employment disputes (discrimination, wrongful termination, wage violations), "
            "customer liability claims, and vendor/contract disputes across all 8 locations (US + EMEA)."
        ),
        "source_description": (
            "Legal complaint PDFs for Casper's Kitchens Inc. Each document is a confidential "
            "attorney-client privileged case file covering case number, parties, legal basis, "
            "applicable statutes, relief sought, case status, and risk assessment."
        ),
        "instructions": (
            "You are a legal intelligence assistant for the CEO of Casper's Kitchens. "
            "Always cite the specific case number and location when answering. "
            "Clearly state the risk level (HIGH/MEDIUM/LOW) and amount at stake. "
            "Remind the CEO that all legal decisions should be made with counsel. "
            "Never speculate on legal outcomes."
        ),
    },
    {
        "name": f"{CATALOG}-ceo-regulatory",
        "endpoint": REG_ENDPOINT,
        "volume_path": f"/Volumes/{CATALOG}/regulatory/documents",
        "source_name": "regulatory_documents",
        "description": (
            "Answers questions about Casper's Kitchens regulatory compliance status. "
            "Covers food service permits, fire safety certificates, zoning compliance, "
            "FDA registrations, and food handler certifications across all 8 locations (US + EMEA)."
        ),
        "source_description": (
            "Regulatory compliance PDFs for all 4 Casper's Kitchens ghost kitchen locations. "
            "Includes document type, issuing authority, issue date, expiry date, current status, "
            "and specific conditions and requirements from each regulatory body."
        ),
        "instructions": (
            "You are a regulatory compliance assistant for the CEO. "
            "Always cite the specific document ID, issuing authority, and expiry date. "
            "Flag any permits or certificates that are expiring within 60 days or have a conditional status. "
            "Summarize compliance risk clearly and concisely."
        ),
    },
    {
        "name": f"{CATALOG}-ceo-audits",
        "endpoint": AUDIT_ENDPOINT,
        "volume_path": f"/Volumes/{CATALOG}/audits/reports",
        "source_name": "audit_reports",
        "description": (
            "Answers CEO-level questions about Casper's Kitchens audit findings. "
            "Covers financial statement audits, operational compliance, food safety management, "
            "and supply chain audits across all locations and quarters."
        ),
        "source_description": (
            "Independent audit reports prepared by Big 4 and mid-tier firms for Casper's Kitchens. "
            "Each report includes audit type, period, scope, auditor's opinion, individual findings "
            "(Critical/Significant/Minor/Informational) with remediation details."
        ),
        "instructions": (
            "You are an audit intelligence assistant for the CEO. "
            "Always cite the specific audit ID, auditing firm, and period covered. "
            "Highlight Critical and Significant findings prominently. "
            "Summarize the auditor's opinion and any patterns across multiple audits. "
            "Be specific about financial impacts and remediation deadlines."
        ),
    },
    {
        "name": f"{CATALOG}-ceo-consultancy",
        "endpoint": CONSULT_ENDPOINT,
        "volume_path": f"/Volumes/{CATALOG}/consultancy/reports",
        "source_name": "consultancy_reports",
        "description": (
            "Answers questions about strategic recommendations from management consultants. "
            "Covers market expansion, operations efficiency, AI transformation, and workforce "
            "management reports from top-tier consulting firms."
        ),
        "source_description": (
            "Management consulting reports prepared exclusively for Casper's Kitchens. "
            "Each report includes executive summary, detailed section-by-section analysis, "
            "financial projections, and specific recommendations with ROI estimates."
        ),
        "instructions": (
            "You are a strategic intelligence assistant for the CEO. "
            "Always cite the consulting firm and report date. "
            "Summarize key recommendations with financial impact figures. "
            "When multiple reports address the same topic, synthesize the consensus view. "
            "Be direct and executive-ready in your summaries."
        ),
    },
]


def _list_kas():
    """Return all KA records from the KA API (handles pagination)."""
    kas = []
    params = {}
    try:
        while True:
            resp = w.api_client.do("GET", API_BASE, query=params)
            for item in resp.get("knowledge_assistants", []):
                kas.append(item)
            token = resp.get("next_page_token")
            if not token:
                break
            params = {"page_token": token}
    except Exception as e:
        print(f"  KA list error: {e}")
    return kas


def _ka_id_from_item(item):
    ka = item.get("knowledge_assistant", item)
    return item.get("tile_id") or ka.get("tile_id") or ka.get("id")


def _ka_id_exists(ka_id):
    """Return True only if the KA API can actually GET this ID."""
    try:
        w.api_client.do("GET", f"{API_BASE}/{ka_id}")
        return True
    except Exception:
        return False


def find_ka_by_endpoint(endpoint_name):
    """Find a live KA tile_id by endpoint_name, skipping stale/deleted entries."""
    for item in _list_kas():
        ka = item.get("knowledge_assistant", item)
        if ka.get("endpoint_name") == endpoint_name:
            kid = _ka_id_from_item(item)
            if kid and _ka_id_exists(kid):
                return kid
    return None


def find_ka_by_name(name):
    """Find a live KA tile_id by name, skipping stale/deleted entries."""
    for item in _list_kas():
        ka = item.get("knowledge_assistant", item)
        if ka.get("name") == name:
            kid = _ka_id_from_item(item)
            if kid and _ka_id_exists(kid):
                return kid
    return None


def _id_from_post(resp):
    """Extract tile_id from a POST /knowledge-assistants response."""
    if isinstance(resp, dict):
        ka = resp.get("knowledge_assistant", resp)
        return resp.get("tile_id") or ka.get("tile_id") or ka.get("id")
    return None


created_agents = []

for cfg in KA_CONFIGS:
    print(f"\n--- Processing KA: {cfg['name']} ---")

    ka_body = {
        "name": cfg["name"],
        "description": cfg["description"],
        "endpoint_name": cfg["endpoint"],
        "knowledge_sources": [
            {
                "files_source": {
                    "name": cfg["source_name"],
                    "type": "files",
                    "files": {"path": cfg["volume_path"]},
                    "description": cfg["source_description"],
                }
            }
        ],
        "instructions": cfg["instructions"],
    }

    agent_id = None
    needs_polling = True

    # 1. Look up by endpoint_name directly in the KA API — most reliable source of truth
    existing_id = find_ka_by_endpoint(cfg["endpoint"])
    if not existing_id:
        existing_id = find_ka_by_name(cfg["name"])

    if existing_id:
        try:
            w.api_client.do("PUT", f"{API_BASE}/{existing_id}", body=ka_body)
            agent_id = existing_id
            print(f"♻️  Updated existing KA: {agent_id}")
        except Exception as e:
            print(f"  PUT failed for {existing_id}: {e} — will create new")

    # 2. Create fresh only if no existing KA found or PUT failed
    if not agent_id:
        try:
            resp = w.api_client.do("POST", API_BASE, body=ka_body)
            # Extract new tile_id directly from the POST response — don't re-query
            # the list API which may still return stale deleted entries.
            agent_id = _id_from_post(resp)
            print(f"✅ Created KA: {agent_id}")
        except Exception as e:
            if "already exists" in str(e).lower():
                print(f"♻️  KA already exists (race condition) — fetching ID")
            else:
                raise

        # Only fall back to list lookup if POST didn't return an ID
        if not agent_id:
            time.sleep(10)
            agent_id = find_ka_by_endpoint(cfg["endpoint"]) or find_ka_by_name(cfg["name"])
        if not agent_id:
            print(f"⚠️  Could not resolve agent_id — will proceed without polling")
            needs_polling = False

    # 3. Poll for readiness
    if needs_polling and agent_id:
        print(f"  Polling KA readiness ({agent_id})...")
        endpoint_ready = False
        for attempt in range(30):
            try:
                status = w.api_client.do("GET", f"{API_BASE}/{agent_id}")
                ka_status = status.get("knowledge_assistant", {}).get("status", {})
                ep_status = ka_status.get("endpoint_status", "")
                sync_status = ka_status.get("sync_status", ka_status.get("indexing_status", ""))
                ep_ready = str(ep_status).upper() in ("ONLINE", "ACTIVE", "READY")
                sync_ready = (not sync_status) or str(sync_status).upper() in ("SYNCED", "READY", "COMPLETE", "SUCCESS", "DONE")
                if ep_ready and sync_ready:
                    print(f"✅ KA is READY (endpoint={ep_status}, sync={sync_status or 'n/a'})")
                    endpoint_ready = True
                    break
                print(f"  [{attempt+1}/30] endpoint={ep_status}, sync={sync_status or 'n/a'}, waiting 30s...")
            except Exception as e:
                print(f"  [{attempt+1}/30] poll error: {e}, waiting 30s...")
            time.sleep(30)
        if not endpoint_ready:
            print(f"⚠️  KA may still be indexing — proceeding")

    add(CATALOG, "knowledge_assistants", {
        "endpoint_name": cfg["endpoint"],
        "tile_id": agent_id,
        "name": cfg["name"],
    })
    created_agents.append({"name": cfg["name"], "tile_id": agent_id, "endpoint": cfg["endpoint"]})
    print(f"   Registered: {cfg['name']} ({agent_id})")

print(f"\n✅ CEO Knowledge Agents stage complete — {len(created_agents)} KAs processed")
